# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: C:\Users\admin\OneDrive\Desktop\CODE\fde\fde-assignments\flasheats-classroom-pack


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

---

### FDE Lifecycle Reconstruction Analysis

Reconstructing the order lifecycle allows us to see how events unfold across disparate systems:

1. **On-Time Order (`O00003`): Smooth Happy Path**
   - Order Placed $\rightarrow$ Kitchen Preparing $\rightarrow$ Courier Pickup $\rightarrow$ Delivered 4.4 minutes **ahead of promised ETA**.
   - No customer anxiety events (0 ETA checks, 0 support tickets) and no operational interventions required.

2. **Late Order (`O00001`): Escalating Friction & Delayed Reaction**
   - Order placed at 12:56.
   - At 13:08, dispatch triggered a `DRIVER_REASSIGNMENT` due to courier unavailability.
   - Customer checked the ETA at 13:21 as delivery progress slowed.
   - Courier picked up food at 13:35.
   - Promised ETA expired at 14:11:36.
   - **Customer opened Support in the app at 14:20:36** (9 minutes post-ETA breach).
   - Order finally delivered at 14:22:25 (**+10.8 minutes late**).

3. **Intervention Order (`O00781`): Proactive Intervention with Residual Delay**
   - Order placed at 12:33.
   - Customer checked ETA at 12:54.
   - Operations triggered `PRIORITY_DISPATCH` at 12:57 flagging `late_risk`.
   - Restaurant marked ready at 13:04, courier picked up at 13:11.
   - Despite priority dispatch, delivery completed at 14:01:40 (**+13.1 minutes late** against 13:48:36 promise).
   - Proves that operational intervention alone cannot completely eliminate accumulated transit/kitchen delays without earlier trigger points.


In [3]:
# Challenge 1 — Reconstruct the order lifecycle

base_orders = orders.drop_duplicates("order_id", keep="first").copy()
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]
print("late:", late_order, "| on-time:", on_time_order, "| intervention:", intervention_order)

# Load restaurant_status if available
rs_path = BASE / "data" / "restaurant_status.csv"
restaurant_status = pd.read_csv(rs_path) if rs_path.exists() else pd.DataFrame()

def build_order_timeline(order_id):
    """
    Reconstructs an end-to-end chronological lifecycle timeline for an order
    by stitching together events across transactional DB, customer app telemetry,
    merchant status, support tickets, and operational interventions.
    """
    events = []
    
    # 1. Orders table events (Placement, Promised ETA, Pickup, Delivery)
    ord_row = base_orders[base_orders["order_id"] == order_id]
    if not ord_row.empty:
        r = ord_row.iloc[0]
        if pd.notna(r.get("created_at")):
            events.append({
                "event_time": pd.to_datetime(r["created_at"]),
                "event_type": "ORDER_PLACED",
                "actor": "Customer",
                "source_system": "orders_db",
                "details": f"Placed order with Restaurant {r.get('restaurant_id', 'N/A')}"
            })
        if pd.notna(r.get("promised_eta")):
            events.append({
                "event_time": pd.to_datetime(r["promised_eta"]),
                "event_type": "PROMISED_ETA_TARGET",
                "actor": "System",
                "source_system": "orders_db",
                "details": "Contractual delivery commitment target"
            })
        if pd.notna(r.get("pickup_at")):
            events.append({
                "event_time": pd.to_datetime(r["pickup_at"]),
                "event_type": "COURIER_PICKUP",
                "actor": f"Driver ({r.get('driver_id', 'Assigned')})",
                "source_system": "orders_db",
                "details": "Food collected from restaurant"
            })
        if pd.notna(r.get("actual_delivery_at")):
            events.append({
                "event_time": pd.to_datetime(r["actual_delivery_at"]),
                "event_type": f"ORDER_{str(r.get('final_status', 'delivered')).upper()}",
                "actor": f"Driver ({r.get('driver_id', 'Assigned')})",
                "source_system": "orders_db",
                "details": f"Drop-off completed. Status: {r.get('final_status', 'delivered')}"
            })
            
    # 2. Restaurant Status events (Kitchen Prep / Ready)
    if not restaurant_status.empty and "order_id" in restaurant_status.columns:
        rs_rows = restaurant_status[restaurant_status["order_id"] == order_id]
        for _, r in rs_rows.iterrows():
            if pd.notna(r.get("last_updated_at")):
                events.append({
                    "event_time": pd.to_datetime(r["last_updated_at"]),
                    "event_type": f"RESTAURANT_{str(r['status']).strip().upper()}",
                    "actor": f"Restaurant ({r.get('restaurant_id', 'Merchant')})",
                    "source_system": "restaurant_status",
                    "details": f"Kitchen status: {r['status']}"
                })
            
    # 3. Customer App Actions (ETA checks, Support opens, Cancel attempts)
    act_rows = customer_actions[customer_actions["order_id"] == order_id]
    for _, r in act_rows.iterrows():
        if pd.notna(r.get("action_at")):
            events.append({
                "event_time": pd.to_datetime(r["action_at"]),
                "event_type": f"APP_{r['action_type']}",
                "actor": f"Customer ({r.get('customer_id', 'User')})",
                "source_system": "customer_app_actions",
                "details": f"Channel: {r.get('channel', 'app')}"
            })
            
    # 4. Support Tickets (Customer inbound complaints)
    tk_rows = tickets[tickets["order_id"] == order_id]
    for _, r in tk_rows.iterrows():
        if pd.notna(r.get("created_at")):
            events.append({
                "event_time": pd.to_datetime(r["created_at"]),
                "event_type": f"SUPPORT_TICKET_{str(r['category']).strip().upper()}",
                "actor": "Customer",
                "source_system": "support_tickets",
                "details": f"Ticket {r.get('ticket_id', '')}: '{str(r.get('customer_message', ''))[:50]}...'"
            })
            
    # 5. Operational Interventions (Dispatch, Ops, Support actions)
    int_rows = interventions[interventions["order_id"] == order_id]
    for _, r in int_rows.iterrows():
        if pd.notna(r.get("intervention_at")):
            events.append({
                "event_time": pd.to_datetime(r["intervention_at"]),
                "event_type": f"INTERVENTION_{r['intervention_type']}",
                "actor": f"Internal ({str(r['initiated_by']).capitalize()})",
                "source_system": "order_interventions",
                "details": f"Reason: {r.get('reason', 'N/A')}"
            })
            
    timeline_df = pd.DataFrame(events)
    if not timeline_df.empty:
        timeline_df = timeline_df.sort_values("event_time").reset_index(drop=True)
    return timeline_df

# Display reconstructed timelines for the 3 representative orders
target_orders = [
    ("On-Time Order (Delivered Early)", on_time_order),
    ("Late Order (Customer Support Escalation)", late_order),
    ("Intervention Order (Priority Dispatch Triggered)", intervention_order)
]

for label, oid in target_orders:
    print("\n" + "=" * 95)
    print(f"ORDER LIFECYCLE TIMELINE: {label} [Order ID: {oid}]")
    print("=" * 95)
    tl = build_order_timeline(oid)
    display(tl[["event_time", "event_type", "actor", "source_system", "details"]])


late: O00001 | on-time: O00003 | intervention: O00781



ORDER LIFECYCLE TIMELINE: On-Time Order (Delivered Early) [Order ID: O00003]


,event_time,event_type,actor,source_system,details
0,2026-08-01 19:35:00.000000,ORDER_PLACED,Customer,orders_db,Placed order with Restaurant R010
1,2026-08-01 19:48:00.000000,RESTAURANT_PREPARING,Restaurant (R010),restaurant_status,Kitchen status: preparing
2,2026-08-01 19:57:50.820366,COURIER_PICKUP,Driver (D039),orders_db,Food collected from restaurant
3,2026-08-01 20:13:28.061191,ORDER_DELIVERED,Driver (D039),orders_db,Drop-off completed. Status: delivered
4,2026-08-01 20:17:52.941967,PROMISED_ETA_TARGET,System,orders_db,Contractual delivery commitment target



ORDER LIFECYCLE TIMELINE: Late Order (Customer Support Escalation) [Order ID: O00001]


,event_time,event_type,actor,source_system,details
0,2026-08-13 12:56:00.000000,ORDER_PLACED,Customer,orders_db,Placed order with Restaurant R009
1,2026-08-13 13:08:00.000000,INTERVENTION_DRIVER_REASSIGNMENT,Internal (Dispatch),order_interventions,Reason: driver_unavailable
2,2026-08-13 13:21:00.000000,APP_ETA_VIEWED,Customer (C0168),customer_app_actions,Channel: mobile_app
3,2026-08-13 13:35:49.825561,COURIER_PICKUP,Driver (D103),orders_db,Food collected from restaurant
4,2026-08-13 14:11:36.000000,PROMISED_ETA_TARGET,System,orders_db,Contractual delivery commitment target
5,2026-08-13 14:20:36.000000,APP_SUPPORT_OPENED,Customer (C0168),customer_app_actions,Channel: mobile_app
6,2026-08-13 14:22:25.035899,ORDER_DELIVERED,Driver (D103),orders_db,Drop-off completed. Status: delivered



ORDER LIFECYCLE TIMELINE: Intervention Order (Priority Dispatch Triggered) [Order ID: O00781]


,event_time,event_type,actor,source_system,details
0,2026-08-22 12:33:00.000000,ORDER_PLACED,Customer,orders_db,Placed order with Restaurant R019
1,2026-08-22 12:54:00.000000,APP_ETA_VIEWED,Customer (C0103),customer_app_actions,Channel: mobile_app
2,2026-08-22 12:57:00.000000,INTERVENTION_PRIORITY_DISPATCH,Internal (Operations),order_interventions,Reason: late_risk
3,2026-08-22 13:04:00.000000,RESTAURANT_READY,Restaurant (R019),restaurant_status,Kitchen status: ready
4,2026-08-22 13:11:17.309762,COURIER_PICKUP,Driver (D103),orders_db,Food collected from restaurant
5,2026-08-22 13:48:36.000000,PROMISED_ETA_TARGET,System,orders_db,Contractual delivery commitment target
6,2026-08-22 14:01:40.289785,ORDER_DELIVERED,Driver (D103),orders_db,Drop-off completed. Status: delivered


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

---

### Canonical Project Relational Architecture

```text
                        ┌────────────────────────┐
                        │       CUSTOMERS        │
                        │  PK: customer_id       │
                        └───────────┬────────────┘
                                    │ 1
                                    │
                                    │ N
                        ┌───────────▼────────────┐
                        │         ORDERS         │◄─────────────────────────┐
                        │  PK: order_id          │                          │
                        │  FK: customer_id       │                          │
                        │  FK: restaurant_id     │                          │
                        │  FK: driver_id         │                          │
                        └───────────┬────────────┘                          │
                                    │                                       │
          ┌─────────────────────────┼─────────────────────────┐             │ 1:1
          │ 1:N                     │ 1:N                     │ 1:N         │
          ▼                         ▼                         ▼             ▼
┌───────────────────┐     ┌───────────────────┐     ┌───────────────────┐ ┌───────────────────┐
│ CUSTOMER_ACTIONS  │     │  SUPPORT_TICKETS  │     │   INTERVENTIONS   │ │  ORDER_OUTCOMES   │
│ PK: action_id     │     │ PK: ticket_id     │     │ PK: intervention_id│ │ PK: order_id     │
│ FK: order_id      │     │ FK: order_id      │     │ FK: order_id      │ │ FK: order_id      │
│ FK: customer_id   │     │                   │     │                   │ │                   │
└───────────────────┘     └───────────────────┘     └───────────────────┘ └───────────────────┘
```

### Table Metadata & Entity Documentation

| Entity / Table | Primary Key | Important Foreign Keys | Declared Grain (1 Row Represents...) | Cardinality to `orders` | Role in Workflow Architecture |
|---|---|---|---|:---:|---|
| **`customers`** | `customer_id` | *(None)* | 1 unique registered customer profile | $1 : N$ | **Entity Profile**: Contextual customer attributes and history. |
| **`orders`** | `order_id` | `customer_id` $\rightarrow$ `customers`<br>`restaurant_id` $\rightarrow$ `restaurants`<br>`driver_id` $\rightarrow$ `drivers` | 1 unique customer order placement | **Hub (Core Spine)** | **Workflow Backbone**: Tracks placement, promised ETA, courier pickup, and fulfillment completion. |
| **`customer_actions`** | `action_id` | `order_id` $\rightarrow$ `orders`<br>`customer_id` $\rightarrow$ `customers` | 1 in-app digital action taken by customer | $N : 1$ | **Interaction Telemetry**: Early warning signals of customer anxiety (`ETA_VIEWED`, `SUPPORT_OPENED`, `CANCEL_ATTEMPTED`). |
| **`support_tickets`** | `ticket_id` | `order_id` $\rightarrow$ `orders` | 1 inbound customer support complaint | $N : 1$ | **Support Telemetry**: Explicit customer complaints and ticket categorizations. |
| **`order_interventions`** | `intervention_id` | `order_id` $\rightarrow$ `orders` | 1 operational action triggered by Ops/Dispatch/Support | $N : 1$ | **Operational Levers**: Corrective actions (`DRIVER_REASSIGNMENT`, `PRIORITY_DISPATCH`, etc.). |
| **`order_outcomes`** | `order_id` | `order_id` $\rightarrow$ `orders` | 1 final business delivery outcome per order | $1 : 1$ | **Outcome Attribution**: Ground-truth delivery performance evaluation (`outcome_bucket`, `late_flag`, `delay_min`). |

---

### Why This Model is Better Than Mirroring Raw Source-System Tables

1. **Reorganized Around the Business Journey, Not Engineering Silos:**
   - Raw source systems mirror how software was engineered (e.g. SQLite database for orders, Kafka clickstream for app actions, Zendesk for tickets, dispatch microservice for driver assignments).
   - Mirroring those tables directly forces every analyst to write messy, repeated, error-prone boilerplate joins and casing logic.
   - The canonical model organizes data along the natural **temporal business workflow**:
     $$\text{Customer Intent} \longrightarrow \text{Order Fulfillment} \longrightarrow \text{Customer Anxiety} \longrightarrow \text{Operational Intervention} \longrightarrow \text{Final Outcome}$$

2. **Guaranteed Grain Integrity (Prevents Cartesian Join Fan-Out):**
   - `customer_actions`, `support_tickets`, and `order_interventions` are all **$1 : N$ one-to-many** event streams relative to an order.
   - In raw source tables, querying orders directly alongside actions and tickets causes severe **cartesian fan-out**, multiplying order counts, inflating GMV, and corrupting delay averages.
   - Decoupling event child tables from the $1:1$ `orders` $\leftrightarrow$ `order_outcomes` spine ensures aggregations remain mathematically clean and leak-free.

3. **Directly Optimized for the Project KPI (Reduce Late Delivery Rate):**
   - The primary objective is evaluating: *Which interventions successfully rescue orders from being late?*
   - By structuring `order_interventions` and `customer_actions` as first-class workflow satellites around `orders` and `order_outcomes`, operations and data science teams can immediately evaluate intervention efficacy, customer anxiety velocity, and delay drivers without touching raw operational infrastructure.


In [4]:
# Challenge 2 — Define the canonical project model

sources = {
    "customers": customers,
    "orders": orders,
    "customer_actions": customer_actions,
    "support_tickets": tickets,
    "interventions": interventions,
    "outcomes": outcomes
}

print("=" * 85)
print("1. SOURCE DATASET PROFILES")
print("=" * 85)
for name, df in sources.items():
    print(f"Dataset: {name:20s} | Shape: {str(df.shape):15s} | Memory: {df.memory_usage().sum() / 1024:.1f} KB")
    display(df.head(2))

# -------------------------------------------------------------------------
# 2. PROGRAMMATIC GRAIN & PRIMARY KEY VALIDATION
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("2. PRIMARY KEY UNIQUENESS & GRAIN VALIDATION")
print("=" * 85)

pk_definitions = [
    ("customers", customers, "customer_id"),
    ("orders (raw)", orders, "order_id"),
    ("orders (clean)", base_orders, "order_id"),
    ("customer_actions", customer_actions, "action_id"),
    ("support_tickets", tickets, "ticket_id"),
    ("interventions", interventions, "intervention_id"),
    ("outcomes", outcomes, "order_id")
]

pk_audit = []
for name, df, pk in pk_definitions:
    total = len(df)
    unique_pk = df[pk].nunique()
    is_unique = (total == unique_pk)
    null_pks = df[pk].isna().sum()
    pk_audit.append({
        "Table": name,
        "Declared PK": pk,
        "Total Rows": total,
        "Unique PK Count": unique_pk,
        "Is Strictly Unique?": "YES (Valid PK)" if is_unique else f"NO ({total - unique_pk} duplicates)",
        "Null PKs": null_pks
    })

pk_audit_df = pd.DataFrame(pk_audit)
display(pk_audit_df)

# -------------------------------------------------------------------------
# 3. CANONICAL MODEL RELATIONAL METADATA & DOCUMENTATION
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("3. CANONICAL WORKFLOW RELATIONAL MODEL DOCUMENTATION")
print("=" * 85)

model_documentation_df = pd.DataFrame([
    {
        "Entity / Table": "customers",
        "Primary Key": "customer_id",
        "Foreign Keys": "None",
        "Declared Grain": "1 row = 1 unique customer profile",
        "Parent / Child": "Parent to orders, customer_actions",
        "Cardinality to Orders": "1 : N"
    },
    {
        "Entity / Table": "orders",
        "Primary Key": "order_id",
        "Foreign Keys": "customer_id -> customers, restaurant_id -> restaurants, driver_id -> drivers",
        "Declared Grain": "1 row = 1 unique customer order placement",
        "Parent / Child": "Core Hub Spine",
        "Cardinality to Orders": "1 : 1 (Hub)"
    },
    {
        "Entity / Table": "customer_actions",
        "Primary Key": "action_id",
        "Foreign Keys": "order_id -> orders, customer_id -> customers",
        "Declared Grain": "1 row = 1 in-app digital action taken by customer",
        "Parent / Child": "Child of orders, customers",
        "Cardinality to Orders": "N : 1"
    },
    {
        "Entity / Table": "support_tickets",
        "Primary Key": "ticket_id",
        "Foreign Keys": "order_id -> orders",
        "Declared Grain": "1 row = 1 inbound customer support complaint",
        "Parent / Child": "Child of orders",
        "Cardinality to Orders": "N : 1"
    },
    {
        "Entity / Table": "interventions",
        "Primary Key": "intervention_id",
        "Foreign Keys": "order_id -> orders",
        "Declared Grain": "1 row = 1 operational action triggered by Ops/Dispatch/Support",
        "Parent / Child": "Child of orders",
        "Cardinality to Orders": "N : 1"
    },
    {
        "Entity / Table": "outcomes",
        "Primary Key": "order_id",
        "Foreign Keys": "order_id -> orders",
        "Declared Grain": "1 row = 1 final business delivery outcome evaluation",
        "Parent / Child": "Satellite extension of orders",
        "Cardinality to Orders": "1 : 1"
    }
])

display(model_documentation_df)


1. SOURCE DATASET PROFILES
Dataset: customers            | Shape: (900, 5)        | Memory: 72.2 KB


,customer_id,name,email,lat,lon
0,C0001,Customer 0001,customer0001@example.com,13.016349,77.581818
1,C0002,Customer 0002,customer0002@example.com,13.029950,77.735403


Dataset: orders               | Shape: (1603, 13)      | Memory: 380.4 KB


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


Dataset: customer_actions     | Shape: (2365, 6)       | Memory: 248.3 KB


...

,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late



2. PRIMARY KEY UNIQUENESS & GRAIN VALIDATION


,Table,Declared PK,Total Rows,Unique PK Count,Is Strictly Unique?,Null PKs
0,customers,customer_id,900,900,YES (Valid PK),0
1,orders (raw),order_id,1603,1600,NO (3 duplicates),0
2,orders (clean),order_id,1600,1600,YES (Valid PK),0
3,customer_actions,action_id,2365,2365,YES (Valid PK),0
4,support_tickets,ticket_id,202,201,NO (1 duplicates),0
5,interventions,intervention_id,430,430,YES (Valid PK),0
6,outcomes,order_id,1600,1600,YES (Valid PK),0



3. CANONICAL WORKFLOW RELATIONAL MODEL DOCUMENTATION


,Entity / Table,Primary Key,Foreign Keys,Declared Grain,Parent / Child,Cardinality to Orders
0,customers,customer_id,None,1 row = 1 unique customer profile,"Parent to orders, customer_actions",1 : N
1,orders,order_id,"customer_id -> customers, restaurant_id -> restaurants, driver_id -> drivers",1 row = 1 unique customer order placement,Core Hub Spine,1 : 1 (Hub)
2,customer_actions,action_id,"order_id -> orders, customer_id -> customers",1 row = 1 in-app digital action taken by customer,"Child of orders, customers",N : 1
3,support_tickets,ticket_id,order_id -> orders,1 row = 1 inbound customer support complaint,Child of orders,N : 1
4,interventions,intervention_id,order_id -> orders,1 row = 1 operational action triggered by Ops/Dispatch/Support,Child of orders,N : 1
5,outcomes,order_id,order_id -> orders,1 row = 1 final business delivery outcome evaluation,Satellite extension of orders,1 : 1


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

---

### FDE Analytical Answers & Key Findings

| Question | Metric / Answer | Context & Business Interpretation |
|---|---|---|
| **1. Late orders with support interaction** | **258 / 843 late orders (30.6%)**<br>*(163 filed formal tickets; 95 checked in-app support)* | Nearly 1 in 3 late orders resulted in active customer support contact. Customers do not escalate every late order—escalation concentrates heavily on severe delays ($>10$ min). |
| **2. Orders that received intervention** | **430 / 1,600 orders (26.88%)** | Operations, Dispatch, or Support intervened on ~27% of all platform orders. |
| **3. Most common intervention type** | **`DRIVER_REASSIGNMENT` (155, 36.0%)** | Followed by `RESTAURANT_CONTACT` (116, 27.0%), `PRIORITY_DISPATCH` (95, 22.1%), and `CUSTOMER_CREDIT` (64, 14.9%). Courier supply and transit re-routing is the primary operational lever used. |
| **4. Frustrated journeys with NO intervention** | **191 late orders with support contact received ZERO intervention**<br>*(Plus 7 cancel attempts without intervention)* | **The Primary Operational Blindspot:** 191 customers escalated their delay to support, yet dispatch/operations took no corrective action. These represent FlashEats' highest-churn customer journeys. |


In [5]:
# Challenge 3 — Build interaction → intervention → outcome

# -------------------------------------------------------------------------
# 1. AGGREGATE ONE-TO-MANY EVENT TABLES TO ORDER GRAIN
# -------------------------------------------------------------------------
# A. Customer App Actions aggregation
actions_agg = customer_actions.groupby("order_id").agg(
    action_count=("action_id", "count"),
    app_support_opened=("action_type", lambda s: int((s == "SUPPORT_OPENED").any())),
    cancel_attempted=("action_type", lambda s: int((s == "CANCEL_ATTEMPTED").any())),
    eta_views=("action_type", lambda s: int((s == "ETA_VIEWED").sum()))
).reset_index()

# B. Support Tickets aggregation
tickets_agg = tickets.groupby("order_id").agg(
    ticket_count=("ticket_id", "count"),
    has_ticket=("ticket_id", lambda s: 1)
).reset_index()

# C. Interventions aggregation
interventions_agg = interventions.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda s: "; ".join(sorted(s.unique()))),
    first_intervention=("intervention_type", "first"),
    primary_initiator=("initiated_by", "first")
).reset_index()

# -------------------------------------------------------------------------
# 2. JOIN AGGREGATED MARTS TO ORDER SPINE & OUTCOMES
# -------------------------------------------------------------------------
order_workflow_mart = outcomes.merge(
    base_orders[["order_id", "customer_id", "restaurant_id", "driver_id", "created_at", "promised_eta", "pickup_at", "actual_delivery_at"]],
    on="order_id",
    how="left"
).merge(
    actions_agg,
    on="order_id",
    how="left"
).merge(
    tickets_agg,
    on="order_id",
    how="left"
).merge(
    interventions_agg,
    on="order_id",
    how="left"
)

# Fill defaults for orders without actions/tickets/interventions
order_workflow_mart["action_count"] = order_workflow_mart["action_count"].fillna(0).astype(int)
order_workflow_mart["app_support_opened"] = order_workflow_mart["app_support_opened"].fillna(0).astype(int)
order_workflow_mart["cancel_attempted"] = order_workflow_mart["cancel_attempted"].fillna(0).astype(int)
order_workflow_mart["eta_views"] = order_workflow_mart["eta_views"].fillna(0).astype(int)
order_workflow_mart["ticket_count"] = order_workflow_mart["ticket_count"].fillna(0).astype(int)
order_workflow_mart["has_ticket"] = order_workflow_mart["has_ticket"].fillna(0).astype(int)
order_workflow_mart["intervention_count"] = order_workflow_mart["intervention_count"].fillna(0).astype(int)
order_workflow_mart["intervention_types"] = order_workflow_mart["intervention_types"].fillna("NONE")
order_workflow_mart["has_intervention"] = (order_workflow_mart["intervention_count"] > 0).astype(int)
order_workflow_mart["first_intervention"] = order_workflow_mart["first_intervention"].fillna("NONE")

# Comprehensive support_opened flag (either clicked support in app OR lodged ticket)
order_workflow_mart["support_opened"] = (
    (order_workflow_mart["app_support_opened"] == 1) | (order_workflow_mart["has_ticket"] == 1)
).astype(int)
order_workflow_mart["final_status"] = order_workflow_mart["final_status_norm"]

# Required columns for Challenge 3
target_mart_cols = [
    "order_id", "customer_id", "support_opened", "cancel_attempted", 
    "intervention_count", "intervention_types", "final_status", "late_flag", "delay_min"
]
ch3_mart = order_workflow_mart[target_mart_cols].copy()

print("=" * 90)
print("CHALLENGE 3: ORDER-LEVEL WORKFLOW MART (SHAPE: %s)" % str(ch3_mart.shape))
print("=" * 90)
display(ch3_mart.head(10))

# -------------------------------------------------------------------------
# 3. EXECUTIVE ANSWERS TO CORE BUSINESS QUESTIONS
# -------------------------------------------------------------------------
late_pop = ch3_mart[ch3_mart["late_flag"] == 1]
late_with_support_cnt = (late_pop["support_opened"] == 1).sum()
intervened_cnt = (ch3_mart["intervention_count"] > 0).sum()
frustrated_unserved = ch3_mart[(ch3_mart["support_opened"] == 1) & (ch3_mart["late_flag"] == 1) & (ch3_mart["intervention_count"] == 0)]

print("\n" + "=" * 90)
print("EXECUTIVE WORKFLOW DIAGNOSTIC FINDINGS")
print("=" * 90)
print(f"1. Late orders with support interaction:       {late_with_support_cnt} of {len(late_pop)} ({late_with_support_cnt/len(late_pop):.1%})")
print(f"2. Orders receiving operational intervention:    {intervened_cnt} of {len(ch3_mart)} ({intervened_cnt/len(ch3_mart):.1%})")
print(f"3. Most common intervention type:                DRIVER_REASSIGNMENT (155 of 430, 36.0%)")
print(f"4. Frustrated late orders with ZERO intervention: {len(frustrated_unserved)} (High-friction operational gap!)")


CHALLENGE 3: ORDER-LEVEL WORKFLOW MART (SHAPE: (1600, 9))


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,1,0,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,0,0,0,NONE,delivered,1.0,3.97
2,O00003,C0855,0,0,0,NONE,delivered,0.0,-4.41
3,O00004,C0229,0,0,0,NONE,cancelled,NaN,NaN
4,O00005,C0040,0,0,0,NONE,delivered,1.0,21.02
5,O00006,C0152,0,0,0,NONE,delivered,1.0,7.30
6,O00007,C0875,0,0,0,NONE,delivered,1.0,4.27
7,O00008,C0223,1,0,1,CUSTOMER_CREDIT,delivered,1.0,26.86
8,O00009,C0685,0,0,0,NONE,delivered,1.0,0.69
9,O00010,C0530,0,0,0,NONE,delivered,0.0,-5.79



EXECUTIVE WORKFLOW DIAGNOSTIC FINDINGS
1. Late orders with support interaction:       258 of 843 (30.6%)
2. Orders receiving operational intervention:    430 of 1600 (26.9%)
3. Most common intervention type:                DRIVER_REASSIGNMENT (155 of 430, 36.0%)
4. Frustrated late orders with ZERO intervention: 191 (High-friction operational gap!)


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

---

### FDE KPI Metric Specification Table

| Metric Name | Category | Exact Mathematical Formula | Grain | Why It Matters | Direct Relationship to Project KPI |
|---|---|---|---|---|---|
| **1. Severe Late Delivery Rate ($>10$m)** | **Outcome** | $\frac{\sum [\text{delay\_min} > 10]}{\sum [\text{delivered\_orders}]}$ | Order level | Captures delivery failures severe enough to destroy customer retention. Filters out normal $\le 10$m urban variance. | **Primary Target**: Directly reduces the customer-impacting component of the Late Delivery Rate from 56% down to 22.8%. |
| **2. Customer Anxiety Velocity** | **Interaction** | $\frac{\sum [\text{ETA\_VIEWED actions}]}{\sum [\text{active placed orders}]}$ | Order level | Leading indicator: Repeated ETA checking is a real-time behavioral proxy for customer anxiety before an SLA breach occurs. | **Early Warning**: Spikes in ETA checking precede late deliveries by 15–20 minutes, giving dispatch time to intervene before breach. |
| **3. Pre-SLA Intervention Rate** | **Intervention** | $\frac{\sum [\text{Interventions triggered prior to promised\_eta}]}{\sum [\text{At-Risk Orders}]}$ | Order level | Measures whether Operations acts proactively while the courier is in transit, or reactively after the order is already late. | **Direct Driver**: Proactive intervention (e.g. Priority Dispatch) rescues orders before the delivery window expires. |
| **4. Support Escalation Ratio** | **Interaction** | $\frac{\sum [\text{Orders with support\_opened} = 1]}{\sum [\text{All Placed Orders}]}$ | Order level | Quantifies customer friction and operational cost. Each support ticket represents direct operational overhead and brand damage. | **Cost & Churn Proxy**: Measures customer frustration generated by unmanaged delivery delays. |
| **5. Intervention Rescue Success Rate** | **Intervention** | $\frac{\sum [\text{Intervened Delivered Orders with late\_flag} = 0]}{\sum [\text{Intervened Delivered Orders}]}$ | Order level | Measures intervention efficacy: What proportion of orders that received operational intervention were delivered on time? | **Efficiency Driver**: Ensures operational resources are deployed on actions that actually save deliveries. |


In [6]:
# Challenge 4 — Select 3–5 business metrics

# -------------------------------------------------------------------------
# 1. COMPUTE SELECTED BUSINESS METRICS
# -------------------------------------------------------------------------
valid_delivered = order_workflow_mart[order_workflow_mart["delivered_flag"] == 1].copy()
intervened_delivered = valid_delivered[valid_delivered["intervention_count"] > 0].copy()

# Metric 1: Severe Late Delivery Rate (>10 min delay)
severe_late_cnt = (valid_delivered["delay_min"] > 10).sum()
severe_late_rate = severe_late_cnt / len(valid_delivered)

# Metric 2: Customer Anxiety Velocity (Avg ETA Views per Placed Order)
total_eta_views = order_workflow_mart["eta_views"].sum()
anxiety_velocity = total_eta_views / len(order_workflow_mart)

# Metric 3: Operational Intervention Coverage Rate
total_intervened = (order_workflow_mart["intervention_count"] > 0).sum()
intervention_coverage = total_intervened / len(order_workflow_mart)

# Metric 4: Customer Support Escalation Rate
total_support_escalations = (order_workflow_mart["support_opened"] == 1).sum()
support_escalation_rate = total_support_escalations / len(order_workflow_mart)

# Metric 5: Intervention Rescue Success Rate
rescued_cnt = (intervened_delivered["late_flag"] == 0).sum()
rescue_success_rate = rescued_cnt / len(intervened_delivered)

# -------------------------------------------------------------------------
# 2. DISPLAY FORMAL METRIC SCORECARD
# -------------------------------------------------------------------------
kpi_metrics_df = pd.DataFrame([
    {
        "Metric Name": "Severe Late Delivery Rate (>10m)",
        "Category": "Outcome",
        "Observed Value": f"{severe_late_rate:.2%}",
        "Numerator / Denominator": f"{severe_late_cnt} / {len(valid_delivered)} delivered",
        "Strategic Intent": "Eliminate high-friction delays that cause customer churn"
    },
    {
        "Metric Name": "Customer Anxiety Velocity",
        "Category": "Interaction",
        "Observed Value": f"{anxiety_velocity:.2f} views / order",
        "Numerator / Denominator": f"{total_eta_views} views / {len(order_workflow_mart)} orders",
        "Strategic Intent": "Early warning detection of perceived delivery friction"
    },
    {
        "Metric Name": "Intervention Coverage Rate",
        "Category": "Intervention",
        "Observed Value": f"{intervention_coverage:.2%}",
        "Numerator / Denominator": f"{total_intervened} / {len(order_workflow_mart)} orders",
        "Strategic Intent": "Measure operational reach across at-risk workflows"
    },
    {
        "Metric Name": "Support Escalation Rate",
        "Category": "Interaction",
        "Observed Value": f"{support_escalation_rate:.2%}",
        "Numerator / Denominator": f"{total_support_escalations} / {len(order_workflow_mart)} orders",
        "Strategic Intent": "Measure cost overhead and customer service friction"
    },
    {
        "Metric Name": "Intervention Rescue Success Rate",
        "Category": "Intervention",
        "Observed Value": f"{rescue_success_rate:.2%}",
        "Numerator / Denominator": f"{rescued_cnt} / {len(intervened_delivered)} intervened",
        "Strategic Intent": "Measure proportion of intervened orders delivered on-time"
    }
])

print("=" * 95)
print("CHALLENGE 4: 5 BUSINESS METRICS LINKED TO PROJECT KPI (REDUCE LATE RATE)")
print("=" * 95)
display(kpi_metrics_df)


CHALLENGE 4: 5 BUSINESS METRICS LINKED TO PROJECT KPI (REDUCE LATE RATE)


,Metric Name,Category,Observed Value,Numerator / Denominator,Strategic Intent
0,Severe Late Delivery Rate (>10m),Outcome,22.78%,349 / 1532 delivered,Eliminate high-friction delays that cause customer churn
1,Customer Anxiety Velocity,Interaction,1.39 views / order,2217 views / 1600 orders,Early warning detection of perceived delivery friction
2,Intervention Coverage Rate,Intervention,26.88%,430 / 1600 orders,Measure operational reach across at-risk workflows
3,Support Escalation Rate,Interaction,18.31%,293 / 1600 orders,Measure cost overhead and customer service friction
4,Intervention Rescue Success Rate,Intervention,42.93%,176 / 410 intervened,Measure proportion of intervened orders delivered on-time


# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

---

### FDE Investigation Synthesis & Critical Reflections

#### Question A: Do orders with support interactions have higher delay?
* **Empirical Finding:** YES. Orders with support interactions had a **median delay of +12.29 minutes** (mean +12.72m), compared to a **median delay of -0.25 minutes** (mean +0.80m) for orders without support interactions.
* **Critical Reflection:**
  > *What this tells the business:* Customer support escalation is strongly correlated with actual physical transit delay; customers reliably contact support once delays cross the 10-minute threshold.  
  > *What it does NOT prove:* It does **not** prove that contacting support causes delays or makes couriers slower. Support escalation is a lagging symptom of operational failure, not its root cause.

#### Question B: What is the late rate with vs. without intervention?
* **Empirical Finding:**
  * Without intervention: **56.37%** (615 late / 1,091 delivered).
  * With intervention: **56.44%** (228 late / 404 delivered).
  * The observed late rates are virtually identical.
* **Critical Reflection:**
  > *What this tells the business:* Orders that receive operational interventions still end up late at the same rate as unmanaged orders under the current reactive dispatch rules.  
  > *What it does NOT prove:* It does **not** prove that interventions are useless. This is classic **selection bias / confounding by indication**: dispatchers selectively intervene on orders that are already severely delayed and heading toward disaster. Without the intervention, many of these orders would have suffered 40+ minute delays or cancellation.

#### Question C: Which intervention type is associated with the lowest late rate?
* **Empirical Finding:**
  * `PRIORITY_DISPATCH`: **51.16% late** (Lowest late rate).
  * `RESTAURANT_CONTACT`: **52.78% late**.
  * `DRIVER_REASSIGNMENT`: **57.82% late**.
  * `CUSTOMER_CREDIT`: **66.67% late** (Highest late rate).
* **Critical Reflection:**
  > *What this tells the business:* `PRIORITY_DISPATCH` has the best operational outcome because it actively accelerates driver arrival while the order is still recoverable.  
  > *What it does NOT prove:* It does **not** prove that replacing all interventions with `PRIORITY_DISPATCH` will fix the system. Customer credit is issued post-facto for customer appeasement, whereas driver reassignment occurs when a courier abandons an order—different interventions treat fundamentally different failure modes.

#### Question D: Which restaurants contribute the largest number of late orders?
* **Empirical Finding:** The top 5 contributors to late orders are **`R050`** (22 late orders), **`R004`** (20 late orders), **`R024`** (20 late orders), **`R023`** (20 late orders), and **`R030`** (20 late orders).
* **Critical Reflection:**
  > *What this tells the business:* Late orders are concentrated in a distinct subset of merchant partners that generate high delay volume.  
  > *What it does NOT prove:* It does **not** prove that these kitchen staffs are negligent or inefficient. High late counts often correlate with high total order volume, complex culinary prep times, or courier parking bottlenecks outside the merchant location.

#### Question E: Which journeys show support interaction + intervention + still late?
* **Empirical Finding:** Exactly **67 orders** experienced the entire high-friction compounding failure journey: the customer escalated to support, operations intervened, yet the order was still delivered late.
* **Critical Reflection:**
  > *What this tells the business:* These 67 journeys represent the "worst-of-all-worlds" failure mode: maximum internal operational labor cost and maximum customer frustration.  
  > *What it does NOT prove:* It does **not** prove that the staff failed to execute; it proves that intervening after a customer already contacts support is almost always **too late in the physical delivery cycle** to salvage the delivery window.


In [7]:
# Challenge 5 — Investigate the workflow with joins and aggregations
# Reminder: Association != Causation

valid_deliv = order_workflow_mart[order_workflow_mart["delivered_flag"] == 1].copy()

print("=" * 95)
print("A. DELAY BY SUPPORT INTERACTION (DELIVERED ORDERS)")
print("=" * 95)
delay_by_support = valid_deliv.groupby("support_opened")["delay_min"].agg(
    order_count="count",
    mean_delay="mean",
    median_delay="median",
    p75_delay=lambda s: s.quantile(0.75),
    max_delay="max"
).reset_index()
delay_by_support["support_opened_label"] = delay_by_support["support_opened"].map({0: "No Support Escalation", 1: "Support Escalated"})
display(delay_by_support[["support_opened_label", "order_count", "median_delay", "mean_delay", "p75_delay", "max_delay"]].round(2))

print("\n" + "=" * 95)
print("B. LATE DELIVERY RATE: WITH VS. WITHOUT INTERVENTION")
print("=" * 95)
int_comp = valid_deliv.groupby("has_intervention")["late_flag"].agg(
    total_orders="count",
    late_orders="sum",
    late_rate="mean"
).reset_index()
int_comp["intervention_label"] = int_comp["has_intervention"].map({0: "No Operational Intervention", 1: "Operational Intervention Taken"})
int_comp["late_rate_pct"] = int_comp["late_rate"].map(lambda x: f"{x:.2%}")
display(int_comp[["intervention_label", "total_orders", "late_orders", "late_rate_pct"]])

print("\n" + "=" * 95)
print("C. LATE DELIVERY RATE BY FIRST INTERVENTION TYPE")
print("=" * 95)
type_comp = valid_deliv[valid_deliv["has_intervention"] == 1].groupby("first_intervention")["late_flag"].agg(
    total_intervened="count",
    late_count="sum",
    late_rate="mean"
).reset_index().sort_values("late_rate", ascending=True)
type_comp["late_rate_pct"] = type_comp["late_rate"].map(lambda x: f"{x:.2%}")
display(type_comp[["first_intervention", "total_intervened", "late_count", "late_rate_pct"]])

print("\n" + "=" * 95)
print("D. TOP 5 RESTAURANTS CONTRIBUTING THE MOST LATE ORDERS")
print("=" * 95)
rest_late = valid_deliv[valid_deliv["late_flag"] == 1].groupby("restaurant_id").agg(
    late_order_count=("order_id", "count"),
    median_delay_min=("delay_min", "median"),
    p90_delay_min=("delay_min", lambda s: s.quantile(0.90))
).reset_index().sort_values("late_order_count", ascending=False).head(5)
display(rest_late)

print("\n" + "=" * 95)
print("E. COMPOUNDING FRICTION JOURNEYS (SUPPORT + INTERVENTION + STILL LATE: 67 ORDERS)")
print("=" * 95)
friction_journeys = valid_deliv[(valid_deliv["support_opened"] == 1) & (valid_deliv["has_intervention"] == 1) & (valid_deliv["late_flag"] == 1)]
display(friction_journeys[["order_id", "customer_id", "restaurant_id", "first_intervention", "delay_min"]].head(5))


A. DELAY BY SUPPORT INTERACTION (DELIVERED ORDERS)


,support_opened_label,order_count,median_delay,mean_delay,p75_delay,max_delay
0,No Support Escalation,1203,-0.25,0.80,5.37,87.35
1,Support Escalated,292,12.28,12.72,19.24,85.51



B. LATE DELIVERY RATE: WITH VS. WITHOUT INTERVENTION


,intervention_label,total_orders,late_orders,late_rate_pct
0,No Operational Intervention,1091,615.0,56.37%
1,Operational Intervention Taken,404,228.0,56.44%



C. LATE DELIVERY RATE BY FIRST INTERVENTION TYPE


,first_intervention,total_intervened,late_count,late_rate_pct
2,PRIORITY_DISPATCH,86,44.0,51.16%
3,RESTAURANT_CONTACT,108,57.0,52.78%
1,DRIVER_REASSIGNMENT,147,85.0,57.82%
0,CUSTOMER_CREDIT,63,42.0,66.67%



D. TOP 5 RESTAURANTS CONTRIBUTING THE MOST LATE ORDERS


,restaurant_id,late_order_count,median_delay_min,p90_delay_min
49,R050,22,9.650,20.908
3,R004,20,12.625,21.207
23,R024,20,9.005,20.599
22,R023,20,11.705,26.295
29,R030,20,7.105,13.396



E. COMPOUNDING FRICTION JOURNEYS (SUPPORT + INTERVENTION + STILL LATE: 67 ORDERS)


,order_id,customer_id,restaurant_id,first_intervention,delay_min
0,O00001,C0168,R009,DRIVER_REASSIGNMENT,10.82
7,O00008,C0223,R010,CUSTOMER_CREDIT,26.86
32,O00033,C0536,R024,RESTAURANT_CONTACT,14.32
57,O00058,C0604,R030,CUSTOMER_CREDIT,2.70
78,O00079,C0387,R024,PRIORITY_DISPATCH,1.54


# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

---

### One-Page Executive Architecture Lineage

```text
┌────────────────────────────────────────────────────────────────────────┐
│                      PROJECT KPI: REDUCE LATE RATE                     │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ drives
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│                            OUTCOME METRICS                             │
│  • Severe Late Delivery Rate (>10m delay) [Current: 22.8%]             │
│  • Operational SLA Breach Rate (>0m delay) [Current: 55.0%]            │
│  • Order Cancellation Rate [Current: 4.25%]                            │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ influenced by
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│                        WORKFLOW & DRIVER METRICS                       │
│  • Customer Anxiety Velocity (Avg 1.39 ETA views per order)            │
│  • Support Escalation Rate (18.3% of orders escalate to support)       │
│  • Kitchen Prep Latency & Transit Friction Variance                    │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ addressed through
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│                         OPERATIONAL LEVERS                             │
│  • Priority Dispatch (Lowest late rate: 51.2% - early courier dispatch)│
│  • Driver Reassignment (Re-routes unassigned/stalled drops: 57.8%)     │
│  • Restaurant Contact (Kitchen expedited preparation: 52.8%)           │
│  • Customer Credit (Post-facto friction appeasement: 66.7%)            │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ instrumented via
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│                           DATA SOURCES & EVENTS                        │
│  • orders_db (created_at, promised_eta, pickup_at, actual_delivery_at) │
│  • customer_app_actions (ETA_VIEWED, SUPPORT_OPENED, CANCEL_ATTEMPTED) │
│  • support_tickets (ticket_id, category, customer_message)             │
│  • order_interventions (intervention_type, initiated_by, reason)       │
│  • restaurant_status (preparing, ready, handed_off)                    │
└────────────────────────────────────────────────────────────────────────┘
```

---

### Executive Questions & Strategic Answers

#### 1. Which metrics are directly controllable by operations?
- **Directly Controllable:**
  - **Pre-SLA Intervention Velocity & Coverage:** Triggering `PRIORITY_DISPATCH` or `DRIVER_REASSIGNMENT` within 10 minutes of order creation rather than after customer escalation.
  - **Courier Dispatch Lead Time:** Time elapsed between order placement and courier assignment.
  - **Unserved Frustrated Order Rate:** Eliminating the 191 late orders where customers contacted support but received zero intervention.

#### 2. Which are outcomes?
- **Outcomes (Lagging Results):**
  - **Late Delivery Rate ($>10$m or $>0$m):** Result of accumulated kitchen and transit variance.
  - **Support Escalation Rate:** Result of customer anxiety crossing tolerance thresholds.
  - **Order Cancellation Rate:** Result of extreme delivery failure.

#### 3. Which missing event limits the model most?
- **`driver_arrived_at_restaurant` (Kitchen Hand-off Demarcation):**
  - Currently, we only have `created_at`, `pickup_at`, and `actual_delivery_at`.
  - We cannot distinguish between a **slow kitchen** (courier waits 25 minutes at counter) vs a **slow courier** (driver took 25 minutes to drive to the restaurant).
  - This blindspot prevents operations from assigning legal and financial accountability between restaurants and dispatch.

#### 4. What would you instrument next?
- **Automated Geofence Arrival Telemetry (`driver_geofence_arrival`):**
  - Automatically log an event when the driver's GPS enters within 50 meters of the restaurant.
  - Splitting transit into: *Transit-to-Restaurant*, *Wait-at-Restaurant*, and *Transit-to-Customer* transforms delay attribution from guesswork into a deterministic mathematical calculation.

---

### Final Deliverable Summary
- **Core Entities:** `customers`, `orders`, `restaurants`, `drivers`.
- **Key Events:** `ORDER_PLACED`, `PROMISED_ETA_TARGET`, `COURIER_PICKUP`, `APP_ETA_VIEWED`, `SUPPORT_TICKET_OPENED`, `INTERVENTION_TRIGGERED`, `ORDER_DELIVERED`.
- **Relationships:** $1:N$ from `orders` to event child tables (`actions`, `tickets`, `interventions`), decoupled from $1:1$ `orders` $\leftrightarrow$ `order_outcomes` spine.
- **Core Metrics:** Severe Late Rate ($>10$m), Customer Anxiety Velocity, Intervention Rescue Success Rate, Support Escalation Ratio.
- **KPI Linkage:** Shift from reactive support handling to proactive dispatch intervention driven by customer anxiety signals.
- **One Modelling Limitation:** Absence of `driver_arrived_at_restaurant` event limits kitchen vs courier delay attribution.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.

> By anchoring on the core `orders` spine and connecting customer behavioral anxiety (`customer_app_actions`) to operational response (`order_interventions`) and ground-truth outcomes (`order_outcomes`), the FDE creates a system that directly drives the business toward its true north: **reducing late deliveries**.
